# IRT Dataset demo

Проверка пакета `irt_data`: кэш → features / temporal → батчи → согласованность аугментаций.

См. также `dataset.md`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch

from irt_data.cache import build_cache
from irt_data.config import DatasetConfig, AugConfig, AugSpec
from irt_data.dataset import IRTDataset
from irt_data.loaders import build_dataloader
from irt_data.transforms import TransformPipeline

ROOT = Path('.').resolve()
print('cwd', ROOT)

## 1. Кэш `.mat` → `.npy` float16

Достаточно пары файлов для демо. Для полного датасета:

```bash
python -m irt_data.cache --sources archive/data --out artifacts/cache
```

In [ ]:
cache_index = build_cache(
    sources=[
        ROOT / 'archive/data/R_002.mat',
        ROOT / 'archive/data/R_003.mat',
        ROOT / 'archive/data/Z_002.mat',
    ],
    out_dir=ROOT / 'artifacts/cache',
    overwrite=False,
)
print('videos in cache:', list(cache_index['videos']))

## 2. Режим `features` → `[C, H, W]`

In [ ]:
cfg_f = DatasetConfig.from_yaml(ROOT / 'configs/features_unet.yaml')
cfg_f.samples_per_video = 2
cfg_f.loader.batch_size = 2

ds_f = IRTDataset(cfg_f)
print('len', len(ds_f), 'ids', ds_f.video_ids)

item = ds_f[0]
print('image', tuple(item['image'].shape), item['image'].dtype)
print('mask ', tuple(item['mask'].shape), 'unique', item['mask'].unique().tolist())

loader_f = build_dataloader(cfg_f, ds_f)
batch_f = next(iter(loader_f))
print('batch image', tuple(batch_f['image'].shape), 'mask', tuple(batch_f['mask'].shape))

# visualize first sample channels + mask
img = batch_f['image'][0].numpy()
msk = batch_f['mask'][0].numpy()
C = img.shape[0]
cols = min(C, 4) + 1
fig, ax = plt.subplots(1, cols, figsize=(3 * cols, 3))
for c in range(cols - 1):
    ax[c].imshow(img[c], cmap='coolwarm')
    ax[c].set_title(f'ch{c}'); ax[c].axis('off')
ax[-1].imshow(msk, cmap='tab10', vmin=0, vmax=5)
ax[-1].set_title('mask'); ax[-1].axis('off')
plt.suptitle('features mode'); plt.tight_layout(); plt.show()

## 3. Режим `temporal` → `[T, C, H, W]`

In [ ]:
cfg_t = DatasetConfig.from_yaml(ROOT / 'configs/temporal_convlstm.yaml')
cfg_t.samples_per_video = 2
cfg_t.temporal.num_frames = 12
cfg_t.temporal.window_size = 12
cfg_t.loader.batch_size = 2

ds_t = IRTDataset(cfg_t)
item_t = ds_t[0]
print('image', tuple(item_t['image'].shape), 'mask', tuple(item_t['mask'].shape))
print('frame_indices', item_t['frame_indices'].tolist())

loader_t = build_dataloader(cfg_t, ds_t)
batch_t = next(iter(loader_t))
print('batch image', tuple(batch_t['image'].shape), '  # [B, T, C, H, W]')

clip = batch_t['image'][0, :, 0].numpy()  # (T, H, W)
msk = batch_t['mask'][0].numpy()
show = [0, len(clip) // 2, len(clip) - 1]
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
for a, t in zip(ax, show):
    a.imshow(clip[t], cmap='inferno')
    a.set_title(f't={t}'); a.axis('off')
ax[-1].imshow(msk, cmap='tab10', vmin=0, vmax=5)
ax[-1].set_title('mask'); ax[-1].axis('off')
plt.suptitle('temporal mode'); plt.tight_layout(); plt.show()

## 4. Аугментации: один геометрический трансформ на весь клип + маску

In [ ]:
tf = TransformPipeline(AugConfig(spatial=[
    AugSpec('HorizontalFlip', {'p': 1.0}),
    AugSpec('VerticalFlip', {'p': 1.0}),
    AugSpec('Affine', {'rotate': [25, 25], 'border_mode': 0, 'p': 1.0}),
]))

T, H, W = 6, 128, 160
frames = np.zeros((T, H, W), np.float32)
frames[:, 40:60, 50:80] = 1.0
mask = np.zeros((H, W), np.uint8)
mask[40:60, 50:80] = 3

out_f, out_m = tf.apply_temporal(frames, mask)

fig, ax = plt.subplots(2, T, figsize=(1.6 * T, 3.2))
for t in range(T):
    ax[0, t].imshow(out_f[t], cmap='gray'); ax[0, t].axis('off'); ax[0, t].set_title(f'f{t}')
    ax[1, t].imshow(out_m, cmap='tab10', vmin=0, vmax=5); ax[1, t].axis('off')
ax[1, 0].set_ylabel('mask')
plt.suptitle('same geometry for all frames + mask')
plt.tight_layout(); plt.show()

centers = []
for t in range(T):
    ys, xs = np.nonzero(out_f[t] > 0.5)
    centers.append((float(ys.mean()), float(xs.mean())))
print('frame centers identical:', len(set(centers)) == 1, centers[0])

## 5. Своя выборка без масок

Для `data/sample*.mat` масок сегментации пока нет — ставьте `mask.missing: zeros` и не считайте mask-loss, либо пропускайте такие сэмплы.

In [ ]:
# раскомментируйте после: python -m irt_data.cache --sources data --out artifacts/cache
# cfg = DatasetConfig.from_yaml('configs/temporal_convlstm.yaml')
# cfg.sources = [type(cfg.sources[0])(root='data', masks=None, pattern='sample*.mat')]
# cfg.mask.missing = 'zeros'
# cfg.temporal.num_frames = 8
# ds = IRTDataset(cfg)
# print(ds.video_ids[:5], ds[0]['image'].shape, bool(ds[0]['has_mask']))
print('see dataset.md for own-sample setup')